# FastAPI Business Report — Research → Analysis → Reporting Pipeline

A multi-agent **Agents SDK** pipeline that hands off across three roles:

1. **Researcher** — gathers facts (web search).
2. **Analyst** — interprets the findings.
3. **Reporter** — writes the final business report.

We stream the run in two phases: **`phase: commentary`** (live progress narration) then **`phase: final_answer`** (the report). We also set **`prompt_cache_retention: 24h`** so repeated runs reuse cached context cheaply.

This notebook is the *driver*; the companion server lives in **`scripts/fastapi_report_server/`** (see its README).

> ## ⚠️ TODO — verify before running
> - Agents SDK import surface is **[unverified]** — see `gpt5-agentic-capabilities.ipynb` for the verification steps (`pip show openai-agents`).
> - `prompt_cache_retention` and the `phase` streaming field names are **[unverified]** against the live changelog.
> - Execution is deferred — students need an API key and the installed SDK.

## Setup

In [ ]:
import os, getpass
def _set_env(var):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")
_set_env("OPENAI_API_KEY")
# !pip install -U openai openai-agents fastapi uvicorn

## 1. Define the three specialist agents

In [ ]:
from agents import Agent  # [unverified] import path

MODEL = "gpt-5.5"

researcher = Agent(
    name="Researcher",
    model=MODEL,
    instructions=(
        "Gather concrete facts and figures relevant to the user's business question. "
        "Use web search. Cite each fact. Do NOT write conclusions."
    ),
    tools=[{"type": "web_search"}],
)

analyst = Agent(
    name="Analyst",
    model=MODEL,
    instructions=(
        "Given the researcher's facts, interpret them: trends, risks, opportunities. "
        "Flag unstated assumptions and any ungrounded numbers. Do NOT format a final report."
    ),
)

reporter = Agent(
    name="Reporter",
    model=MODEL,
    instructions=(
        "Write a concise executive business report from the analyst's interpretation. "
        "Structure: Summary, Key Findings, Risks, Recommendation. <= 1 page."
    ),
)

# Handoff chain: researcher -> analyst -> reporter
analyst.handoffs = [reporter]          # [unverified] handoffs API
researcher.handoffs = [analyst]

## 2. Run the pipeline, streaming `commentary` then `final_answer`

We iterate streamed events and bucket them by **`phase`**. Commentary shows the pipeline thinking out loud; the final answer is the report itself.

In [ ]:
import asyncio
from agents import Runner  # [unverified]

async def run_report(question: str):
    result = Runner.run_streamed(
        researcher,
        question,
        # [unverified] knobs -- confirm exact names against the SDK/changelog:
        run_config={
            "prompt_cache_retention": "24h",   # reuse cached context for 24h
        },
    )

    commentary, final = [], []
    async for event in result.stream_events():
        phase = getattr(event, "phase", None)           # commentary | final_answer
        text = getattr(event, "delta", "") or ""
        if phase == "commentary":
            commentary.append(text)
            print(f"[commentary] {text}", end="", flush=True)
        elif phase == "final_answer":
            final.append(text)

    print("\n\n===== FINAL REPORT =====")
    print("".join(final) or result.final_output)

# await run_report("Should a mid-size SaaS company expand into the EU market in 2026?")
print("Pipeline defined. Uncomment the await call to run.")

## 3. Companion server

A FastAPI server that exposes this pipeline as an HTTP endpoint lives in **`scripts/fastapi_report_server/`**. See **`scripts/fastapi_report_server/README.md`** for setup and run instructions. The notebook is for interactive exploration; the server is how you'd deploy the same pipeline.